# Build Constructor Standings
## Sources

1. fact_session_results
2. dim_constructors

## Output Columns

1. season
2. constructor id
3. constructor name
4. nationality
5. race starts
6. total points
7. number of wins
8. number of podiums
9. standing position

In [0]:
CREATE OR REPLACE VIEW formula1.gold.v_constructor_standing AS
    WITH constructor_session_summary AS (
        SELECT
            f.season,
            c.constructor_id,
            c.constructor_name,
            c.nationality,
            COUNT(*) AS race_starts,
            SUM(f.points) AS total_points,
            COUNT_IF(f.is_win) AS number_of_wins,
            COUNT_IF(f.is_podium) AS number_of_podiums
        FROM
            formula1.gold.fact_session_results f
        INNER JOIN
            formula1.gold.dim_constructors c
        ON
            f.constructor_id == c.constructor_id
        GROUP BY
            f.season,
            c.constructor_id,
            c.constructor_name,
            c.nationality
    )

    SELECT
        season,
        constructor_id,
        constructor_name,
        nationality,
        RANK() OVER(PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS standing,
        race_starts,
        total_points,
        number_of_wins,
        number_of_podiums
    FROM
        constructor_session_summary;